# HumanInputTool Inside a Skill — Meeting Notes Summarizer

This notebook shows that HITL is not only an agent-level concern — it can
be embedded directly inside a reusable skill.

The `meeting-notes-summarizer` skill uses `HumanInputTool` with constrained
`choices` to ask the operator two questions before producing output:

1. **Format** — `bullet-points` or `prose`
2. **Level of detail** — `brief` or `detailed`

Any agent that loads this skill gets the human-input behaviour automatically,
without the caller knowing or configuring it.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama
installed on your local machine and have its LLM hosting service running.
To download Ollama, follow the instructions found on this page:
https://ollama.com/download. After downloading and installing Ollama, you
can start a service by opening a terminal and running `ollama serve`.

In [ ]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"\u2713 Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"\u2713 Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


ensure_ollama()

In [ ]:
import logging

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.tools.default import HumanInputTool

enable_console_logging(logging.INFO)

human_input_tool = HumanInputTool()
llm = OllamaLLM(model="qwen3:14b", think=False)
agent = LLMAgent(llm=llm, tools=[human_input_tool])

In [ ]:
TRANSCRIPT = """\
Sprint 14 Planning — 2026-06-27

Attendees: Priya (PM), Marcus (Lead Eng), Leila (Backend),
Tom (Frontend), Ana (QA)

Priya opened by reviewing Sprint 13 velocity: 42 points delivered against a
target of 45. Two stories were rolled over — the pagination refactor (8 pts)
and the email digest feature (5 pts). Both are high priority for Sprint 14.

Marcus flagged a dependency risk: the pagination refactor depends on a database
migration scheduled for next Tuesday. If the migration slips, the refactor
cannot be merged. The team agreed to move the migration to Monday to de-risk.

Leila volunteered to pick up the email digest feature and estimated three days
of backend work. Tom said he'd need one day of frontend work once Leila's API
is ready. Ana will write the test plan today so it's ready for review by
Wednesday.

Sprint 14 commitments (44 points total):
- Pagination refactor: 8 pts — Marcus, Leila
- Email digest: 5 pts — Leila, Tom
- Auth token refresh: 3 pts — Leila
- Dashboard chart improvements: 13 pts — Tom
- Automated regression suite: 8 pts — Ana
- Performance monitoring alerts: 7 pts — Marcus

Priya reminded the team that the release is scheduled for July 11. All Sprint 14
work must be merged and QA-approved by July 9.

Meeting closed at 10:24.
"""

## Running the Skill

When the agent activates the `meeting-notes-summarizer` skill, it will pause
twice to collect your preferences via `HumanInputTool`:

1. **Format** — `bullet-points` or `prose`
2. **Detail level** — `brief` or `detailed`

The summary is then generated according to your choices.

In [ ]:
result = await agent.run_with_skill(
    "meeting-notes-summarizer",
    prompt=f"Summarise the following meeting transcript:\n\n{TRANSCRIPT}",
)
print(result.content)